# Fix: remove Python docstrings from model inputs

In CodeSearchNet's Python subset the function code includes its own docstring, which is also the target text. The first model run therefore learned to copy the answer from the input (98.6% of Python inputs leaked the target). This notebook strips the docstring from Python inputs with the `ast` module, drops the few rows where the leak remains, and writes `combined_dataset_v2.parquet` and `splits_v2/` (same seed and 80/10/10 ratios as v1).

Run after `01_build_dataset.ipynb`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # ast.parse emits many harmless SyntaxWarnings for regex strings inside dataset code

from google.colab import drive
drive.mount('/content/drive')

import ast, re, json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path('/content/drive/MyDrive/capstone/data')
df = pd.read_parquet(DATA_DIR / 'combined_dataset_v1.parquet')

# Fallback for code that ast cannot parse (e.g. Python 2 syntax)
DOCSTRING_RE = re.compile(
    r'^(\s*(?:async\s+)?def\s.*?:[ \t]*\n)\s*[rRuUbB]{0,2}("""|\'\'\').*?\2[ \t]*\n?',
    re.S)

def strip_python_docstring(src):
    try:
        tree = ast.parse(src)
        fn = tree.body[0]
        if (isinstance(fn, (ast.FunctionDef, ast.AsyncFunctionDef)) and fn.body
                and isinstance(fn.body[0], ast.Expr)
                and isinstance(fn.body[0].value, ast.Constant)
                and isinstance(fn.body[0].value.value, str)):
            node = fn.body[0]
            lines = src.splitlines(keepends=True)
            if lines[node.lineno - 1][:node.col_offset].strip() == '':   # docstring starts on its own line
                return ''.join(lines[:node.lineno - 1] + lines[node.end_lineno:])
        return src
    except SyntaxError:
        return DOCSTRING_RE.sub(r'\1', src, count=1)

def leaks(row):
    ref = row['target_text'].strip()
    if not ref:
        return False
    first = ref.splitlines()[0].strip()[:60]
    return len(first) >= 10 and first in row['input_text']

is_py = (df['language'] == 'python') & (df['source'] == 'codesearchnet')

print('Leak rate BEFORE stripping:')
print(df.assign(leak=df.apply(leaks, axis=1)).groupby('language')['leak'].mean().round(3))

df.loc[is_py, 'input_text'] = df.loc[is_py, 'input_text'].map(strip_python_docstring)

df['leak'] = df.apply(leaks, axis=1)
print('\nLeak rate AFTER stripping:')
print(df.groupby('language')['leak'].mean().round(3))

# Drop any Python rows where stripping failed and the docstring is still in the input
n_before = len(df)
df = df[~(df['leak'] & is_py)].drop(columns='leak').reset_index(drop=True)
print(f'\nDropped {n_before - len(df):,} Python rows still leaking | final size: {len(df):,}')

print('\nSample stripped Python input:\n', df[df['language'] == 'python'].iloc[0]['input_text'][:400])

# Save v2 and re-split with the same seed and ratios as v1
df.to_parquet(DATA_DIR / 'combined_dataset_v2.parquet', index=False)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

SPLIT_DIR = DATA_DIR / 'splits_v2'
SPLIT_DIR.mkdir(exist_ok=True)
train_df.to_parquet(SPLIT_DIR / 'train.parquet', index=False)
val_df.to_parquet(SPLIT_DIR / 'val.parquet', index=False)
test_df.to_parquet(SPLIT_DIR / 'test.parquet', index=False)

with open(DATA_DIR / 'dataset_filter_log_v2.json', 'w') as f:
    json.dump({'base': 'combined_dataset_v1.parquet',
               'change': 'python docstrings stripped from input_text; rows still leaking dropped',
               'rows_dropped': n_before - len(df), 'final_total': len(df), 'split_seed': 42}, f, indent=2)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
